In [ ]:
import gc
import os
import re
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

tqdm.pandas()

ROOT_DIR = Path("/kaggle/input/datasets/glitchr/eurodata")
OUT_DIR = Path("/kaggle/working/output")
RGB_DIR = ROOT_DIR / "EuroSAT_RGB"
MS_DIR = ROOT_DIR / "EuroSAT_MS"
SAR_DIR = ROOT_DIR / "EuroSAT-SAR"

OUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

CLASS_NAMES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"
]
CLASS_IDX = {label: index for index, label in enumerate(CLASS_NAMES)}

datasets = [
    {
        "name": "EuroSAT-SAR",
        "url": "https://huggingface.co/datasets/wangyi111/EuroSAT-SAR/resolve/main/EuroSAT-SAR.zip?download=true",
    },
    {
        "name": "EuroSAT_RGB",
        "url": "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip?download=1",
    },
    {
        "name": "EuroSAT_MS",
        "url": "https://zenodo.org/records/7711810/files/EuroSAT_MS.zip?download=1",
    },
]

for dataset in datasets:
    zip_path = ROOT_DIR / f"{dataset['name']}.zip"
    extract_dir = ROOT_DIR / dataset["name"]

    if not zip_path.exists():
        print(f"Downloading {dataset['name']}...")
        !wget -c "{dataset['url']}" -O "{zip_path}"
    else:
        print(f"{zip_path} already exists. Skipping download.")

    if not extract_dir.exists():
        print(f"Extracting {dataset['name']}...")
        with zipfile.ZipFile(zip_path, "r") as zip_file:
            zip_file.extractall(ROOT_DIR)
    else:
        print(f"{extract_dir} already exists. Skipping extraction.")

print("Dataset preparation completed.")


In [ ]:
def extract_file_id(file_path):
    """Return the numeric file ID and class label from an image path."""
    numbers = re.findall(r"\d+", file_path.stem)
    if not numbers:
        return None, file_path.parent.name

    return int(numbers[-1]), file_path.parent.name


def collect_modality_files(root_dir, modality_name):
    records = []

    for label_folder in sorted(root_dir.iterdir()):
        if not label_folder.is_dir():
            continue

        image_files = sorted(
            file_path
            for file_path in label_folder.rglob("*")
            if file_path.is_file() and file_path.suffix.lower() in VALID_EXTENSIONS
        )

        for image_path in image_files:
            file_id, label = extract_file_id(image_path)
            records.append(
                {
                    "file_id": file_id,
                    "label": label,
                    f"{modality_name}_id": image_path.stem,
                    f"{modality_name}_path": str(image_path),
                }
            )

    return pd.DataFrame(records)


In [ ]:
rgb_df = collect_modality_files(RGB_DIR, "EuroSAT_RGB")

# Enable these when multispectral and SAR modalities are required.
# ms_df = collect_modality_files(MS_DIR, "EuroSAT_MS")
# sar_df = collect_modality_files(SAR_DIR, "EuroSAT_SAR")
# df = (
#     rgb_df.merge(ms_df, on=["file_id", "label"], how="outer")
#     .merge(sar_df, on=["file_id", "label"], how="outer")
# )

df = rgb_df.copy()
print("RGB files:", len(rgb_df))


In [ ]:
final_df = df.dropna(subset=["file_id", "label"]).copy()
final_df["file_id"] = final_df["file_id"].astype(int)
final_df["target"] = final_df["label"].map(CLASS_IDX).astype(int)

final_df = final_df.drop_duplicates(
    subset=["file_id", "label"]
).reset_index(drop=True)

print("Final dataframe shape:", final_df.shape)
display(final_df.head())


In [ ]:
train_df, temp_df = train_test_split(
    final_df,
    test_size=0.30,
    random_state=42,
    stratify=final_df["label"],
    shuffle=True,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"],
    shuffle=True,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


# Data Pipeline

In [ ]:
import tensorflow as tf

IMAGE_SIZE = 64
IMAGE_CHANNELS = 3
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE
RANDOM_SEED = 42
NUM_CLASSES = final_df["label"].nunique()

print("Number of classes:", NUM_CLASSES)


In [ ]:
def load_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=IMAGE_CHANNELS)
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    image = tf.cast(image, tf.float32)

    label = tf.cast(label, tf.int32)
    label = tf.one_hot(label, depth=NUM_CLASSES)
    return image, label


def preprocess_image(image_path, label, augment=False):
    image, label = load_image(image_path, label)

    if augment:
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_flip_up_down(image)
        image = tf.image.random_brightness(image, max_delta=0.1)

    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label


def get_data(
    dataframe,
    batch_size=BATCH_SIZE,
    augment=False,
    shuffle=False,
    repeat=False,
    drop_remainder=False,
):
    image_paths = dataframe["EuroSAT_RGB_path"].to_numpy()
    targets = dataframe["target"].to_numpy(dtype=np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, targets))

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.map(
        lambda path, label: preprocess_image(path, label, augment),
        num_parallel_calls=AUTOTUNE,
    )

    if repeat:
        dataset = dataset.repeat()

    dataset = dataset.batch(batch_size, drop_remainder=drop_remainder)
    return dataset.prefetch(AUTOTUNE)


In [ ]:
sample_ds = get_data(train_df, augment=True, shuffle=True)


In [ ]:
plt.figure(figsize=(10, 10))

for images, labels in sample_ds.take(1):
    for index in range(9):
        plt.subplot(3, 3, index + 1)
        plt.imshow(tf.cast(images[index], tf.uint8))

        label_id = int(tf.argmax(labels[index]).numpy())
        plt.title(CLASS_NAMES[label_id])
        plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
train_ds = get_data(train_df, augment=True, shuffle=True)
val_ds = get_data(val_df)
test_ds = get_data(test_df)


In [ ]:
INPUT_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, IMAGE_CHANNELS)
EPOCHS = 20
LEARNING_RATE = 1e-3

MODEL_DIR = Path("/kaggle/working/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Number of classes:", NUM_CLASSES)
print("Classes:", CLASS_NAMES)


In [14]:
MODEL_REGISTRY = {
    "MobileNetV2": tf.keras.applications.MobileNetV2,
    "MobileNetV3Large": tf.keras.applications.MobileNetV3Large,
    "ResNet50": tf.keras.applications.ResNet50,
    "ResNet101": tf.keras.applications.ResNet101,
    "DenseNet121": tf.keras.applications.DenseNet121,
    "EfficientNetB0": tf.keras.applications.EfficientNetB0,
    "EfficientNetB3": tf.keras.applications.EfficientNetB3,
    "EfficientNetV2B0": tf.keras.applications.EfficientNetV2B0,
    "Xception": tf.keras.applications.Xception,
    "InceptionV3": tf.keras.applications.InceptionV3,
    "ConvNeXtTiny": tf.keras.applications.ConvNeXtTiny,
    "VGG16": tf.keras.applications.VGG16
}

In [ ]:
PREPROCESS_REGISTRY = {
    "MobileNetV2": tf.keras.applications.mobilenet_v2.preprocess_input,
    "ResNet50": tf.keras.applications.resnet.preprocess_input,
    "ResNet101": tf.keras.applications.resnet.preprocess_input,
    "DenseNet121": tf.keras.applications.densenet.preprocess_input,
    "Xception": tf.keras.applications.xception.preprocess_input,
    "InceptionV3": tf.keras.applications.inception_v3.preprocess_input,
    "VGG16": tf.keras.applications.vgg16.preprocess_input,
}


In [ ]:
def get_model(
    model_name,
    input_shape=INPUT_SHAPE,
    num_classes=NUM_CLASSES,
    dropout_rate=0.30,
    base_trainable=False,
):
    if model_name not in MODEL_REGISTRY:
        available_models = ", ".join(MODEL_REGISTRY)
        raise ValueError(
            f"Unknown model: {model_name}. Available models: {available_models}"
        )

    inputs = tf.keras.layers.Input(shape=input_shape, name="optical_image")
    x = inputs

    preprocess_function = PREPROCESS_REGISTRY.get(model_name)
    if preprocess_function is not None:
        x = tf.keras.layers.Lambda(
            preprocess_function,
            name=f"{model_name}_preprocessing",
        )(x)

    base_model = MODEL_REGISTRY[model_name](
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
        pooling="avg",
    )
    base_model.trainable = base_trainable

    x = base_model(x, training=False)
    x = tf.keras.layers.BatchNormalization(name="feature_batch_norm")(x)
    x = tf.keras.layers.Dropout(dropout_rate, name="feature_dropout")(x)
    x = tf.keras.layers.Dense(256, activation="relu", name="feature_dense")(x)
    x = tf.keras.layers.Dropout(dropout_rate, name="classifier_dropout")(x)

    outputs = tf.keras.layers.Dense(
        num_classes,
        activation="softmax",
        name="classification_output",
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name=f"{model_name}_EuroSAT",
    )
    return model, base_model


In [ ]:
def train_single_model(
    model_name,
    train_dataset,
    validation_dataset,
    test_dataset,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
):
    print("\n" + "=" * 70)
    print(f"Training model: {model_name}")
    print("=" * 70)

    tf.keras.backend.clear_session()
    gc.collect()

    model, _ = get_model(
        model_name=model_name,
        input_shape=INPUT_SHAPE,
        num_classes=NUM_CLASSES,
        dropout_rate=0.30,
        base_trainable=False,
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
            tf.keras.metrics.TopKCategoricalAccuracy(
                k=5,
                name="top_5_accuracy",
            ),
        ],
    )

    model.summary()
    model_path = MODEL_DIR / f"{model_name}_best.keras"

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(model_path),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=2,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

    start_time = time.time()
    history = model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1,
    )
    training_time = time.time() - start_time

    test_metrics = model.evaluate(test_dataset, return_dict=True, verbose=1)

    result = {
        "model": model_name,
        "parameters": model.count_params(),
        "best_train_accuracy": max(history.history["accuracy"]),
        "best_validation_accuracy": max(history.history["val_accuracy"]),
        "test_accuracy": test_metrics["accuracy"],
        "test_top_5_accuracy": test_metrics["top_5_accuracy"],
        "test_loss": test_metrics["loss"],
        "training_time_seconds": training_time,
    }

    return model, history, result


In [20]:
MODELS_TO_COMPARE = [
    "DenseNet121",
    # "ResNet50",
    # "EfficientNetB0",
    # "MobileNetV2",
    # "Xception",
    # "ConvNeXtTiny"
]

In [ ]:
comparison_results = []
training_histories = {}

for model_name in MODELS_TO_COMPARE:
    trained_model, model_history, result = train_single_model(
        model_name=model_name,
        train_dataset=train_ds,
        validation_dataset=val_ds,
        test_dataset=test_ds,
        epochs=15,
        learning_rate=1e-3,
    )

    comparison_results.append(result)
    training_histories[model_name] = model_history.history

    del trained_model
    gc.collect()
    tf.keras.backend.clear_session()


In [ ]:
comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values(
    by="test_accuracy",
    ascending=False,
).reset_index(drop=True)

display(comparison_df)


In [ ]:
comparison_path = Path("/kaggle/working/model_comparison_results.csv")
comparison_df.to_csv(comparison_path, index=False)

print(f"Comparison results saved to: {comparison_path}")


In [ ]:
def plot_training_history(histories, model_name):
    history_df = pd.DataFrame(histories[model_name])

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["accuracy"], label="Training Accuracy")
    plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{model_name} Accuracy")
    plt.legend()
    plt.grid()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["loss"], label="Training Loss")
    plt.plot(history_df["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} Loss")
    plt.legend()
    plt.grid()
    plt.show()


In [29]:
MODEL_NAME = 'DenseNet121'

In [ ]:
plot_training_history(training_histories, MODEL_NAME)


In [ ]:
MODEL_PATH = MODEL_DIR / f"{MODEL_NAME}_best.keras"

trained_model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False,
    safe_mode=False,
)
trained_model.summary()


In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)


In [ ]:
predictions = trained_model.predict(test_ds, verbose=1)
predicted_classes = predictions.argmax(axis=1)
true_classes = test_df["target"].to_numpy()


In [ ]:
print(
    classification_report(
        true_classes,
        predicted_classes,
        target_names=CLASS_NAMES,
    )
)


In [ ]:
confusion = confusion_matrix(true_classes, predicted_classes)
confusion


In [ ]:
display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=CLASS_NAMES,
)

display_matrix.plot(xticks_rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
embedding_model = tf.keras.Model(
    inputs=trained_model.input,
    outputs=trained_model.get_layer("feature_dense").output,
    name="optical_embedding_model",
)


In [ ]:
def normalize_embeddings(embeddings):
    return tf.math.l2_normalize(embeddings, axis=1)


In [ ]:
retrieval_df = test_df.dropna(
    subset=["EuroSAT_RGB_path", "label"]
).reset_index(drop=True)

print("Gallery images:", len(retrieval_df))


In [ ]:
gallery_embeddings = embedding_model.predict(test_ds, verbose=1)
gallery_embeddings = normalize_embeddings(gallery_embeddings).numpy()

print("Gallery embedding shape:", gallery_embeddings.shape)


In [ ]:
def extract_query_embedding(image_path, model):
    image, _ = load_image(tf.constant(str(image_path)), tf.constant(0))
    image = tf.expand_dims(image, axis=0)

    embedding = model(image, training=False)
    embedding = tf.math.l2_normalize(embedding, axis=1)
    return embedding.numpy()[0]


In [ ]:
def retrieve_same_modal_query(
    query_index,
    dataframe,
    embeddings,
    top_k=5,
):
    query_embedding = embeddings[query_index]
    similarities = embeddings @ query_embedding
    similarities[query_index] = -np.inf

    ranked_indices = np.argsort(similarities)[::-1][:top_k]
    query_label = dataframe.iloc[query_index]["label"]
    results = []

    for rank, gallery_index in enumerate(ranked_indices, start=1):
        gallery_row = dataframe.iloc[gallery_index]
        retrieved_label = gallery_row["label"]

        results.append(
            {
                "rank": rank,
                "gallery_index": int(gallery_index),
                "file_id": gallery_row["file_id"],
                "label": retrieved_label,
                "image_path": gallery_row["EuroSAT_RGB_path"],
                "similarity": float(similarities[gallery_index]),
                "relevant": int(retrieved_label == query_label),
            }
        )

    return pd.DataFrame(results)


In [ ]:
def retrieve_similar_images(
    query_path,
    gallery_df,
    gallery_embeddings,
    model,
    top_k=5,
    exclude_query=True,
):
    query_embedding = extract_query_embedding(query_path, model)
    similarities = gallery_embeddings @ query_embedding
    ranked_indices = np.argsort(similarities)[::-1]

    results = []

    for gallery_index in ranked_indices:
        gallery_row = gallery_df.iloc[gallery_index]
        gallery_path = gallery_row["EuroSAT_RGB_path"]

        if exclude_query and str(gallery_path) == str(query_path):
            continue

        results.append(
            {
                "rank": len(results) + 1,
                "gallery_index": int(gallery_index),
                "file_id": gallery_row["file_id"],
                "label": gallery_row["label"],
                "image_path": gallery_path,
                "similarity": float(similarities[gallery_index]),
            }
        )

        if len(results) >= top_k:
            break

    return pd.DataFrame(results)


In [ ]:
def display_retrieval_results(
    query_path,
    query_label,
    retrieval_results,
):
    total_images = len(retrieval_results) + 1
    plt.figure(figsize=(4 * total_images, 4))

    query_image = Image.open(query_path).convert("RGB")
    plt.subplot(1, total_images, 1)
    plt.imshow(query_image)
    plt.title(f"Query\n{query_label}")
    plt.axis("off")

    for position, row in retrieval_results.iterrows():
        retrieved_image = Image.open(row["image_path"]).convert("RGB")

        plt.subplot(1, total_images, position + 2)
        plt.imshow(retrieved_image)
        plt.title(
            f"Rank {int(row['rank'])}\n"
            f"{row['label']}\n"
            f"Similarity: {row['similarity']:.3f}"
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
def display_retrieval_grid(
    query_path,
    query_label,
    retrieval_results,
    columns=4,
):
    total_images = len(retrieval_results) + 1
    rows = int(np.ceil(total_images / columns))
    plt.figure(figsize=(4 * columns, 4 * rows))

    query_image = Image.open(query_path).convert("RGB")
    plt.subplot(rows, columns, 1)
    plt.imshow(query_image)
    plt.title(f"Query\n{query_label}")
    plt.axis("off")

    for position, row in retrieval_results.iterrows():
        image = Image.open(row["image_path"]).convert("RGB")

        plt.subplot(rows, columns, position + 2)
        plt.imshow(image)
        plt.title(
            f"Rank {int(row['rank'])}\n"
            f"{row['label']}\n"
            f"{row['similarity']:.3f}"
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
def calculate_query_metrics(
    query_index,
    retrieval_results,
    dataframe,
):
    query_label = dataframe.iloc[query_index]["label"]
    total_relevant = int((dataframe["label"] == query_label).sum() - 1)
    retrieved_relevant = int(retrieval_results["relevant"].sum())
    k = len(retrieval_results)

    precision_at_k = retrieved_relevant / k if k else 0.0
    recall_at_k = (
        retrieved_relevant / total_relevant
        if total_relevant > 0
        else 0.0
    )
    f1_at_k = (
        2 * precision_at_k * recall_at_k / (precision_at_k + recall_at_k)
        if precision_at_k + recall_at_k > 0
        else 0.0
    )

    return {
        "query_index": query_index,
        "query_label": query_label,
        "k": k,
        "retrieved_relevant": retrieved_relevant,
        "total_relevant": total_relevant,
        "precision_at_k": precision_at_k,
        "recall_at_k": recall_at_k,
        "f1_at_k": f1_at_k,
    }


In [ ]:
all_metrics = []

for query_index in tqdm(range(len(test_df))):
    top5_results = retrieve_same_modal_query(
        query_index=query_index,
        dataframe=test_df,
        embeddings=gallery_embeddings,
        top_k=5,
    )

    metrics_at_5 = calculate_query_metrics(
        query_index=query_index,
        retrieval_results=top5_results,
        dataframe=test_df,
    )
    all_metrics.append(metrics_at_5)


In [ ]:
metric_results_df = pd.DataFrame(all_metrics)
display(metric_results_df.head())


In [ ]:
print("Mean Precision@5:", metric_results_df["precision_at_k"].mean())
print("Mean Recall@5:", metric_results_df["recall_at_k"].mean())
print("Mean F1@5:", metric_results_df["f1_at_k"].mean())


In [ ]:
def retrieval_metrics_at_k(
    query_index,
    dataframe,
    embeddings,
    k=5,
    label_column="label",
):
    query_embedding = embeddings[query_index]
    similarities = embeddings @ query_embedding
    similarities[query_index] = -np.inf

    top_k_indices = np.argsort(similarities)[::-1][:k]
    query_label = dataframe.iloc[query_index][label_column]
    retrieved_labels = dataframe.iloc[top_k_indices][label_column].to_numpy()

    relevant_retrieved = int(np.sum(retrieved_labels == query_label))
    total_relevant = int(
        np.sum(dataframe[label_column].to_numpy() == query_label) - 1
    )

    precision = relevant_retrieved / k if k else 0.0
    recall = (
        relevant_retrieved / total_relevant
        if total_relevant > 0
        else 0.0
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "query_index": query_index,
        "query_label": query_label,
        f"precision_at_{k}": precision,
        f"recall_at_{k}": recall,
        f"f1_at_{k}": f1,
        "top_k_indices": top_k_indices,
    }


In [ ]:
def evaluate_image_retrieval(
    dataframe,
    embeddings,
    k_values=(5, 10),
    label_column="label",
):
    rows = []
    retrieval_times = []
    labels = dataframe[label_column].to_numpy()
    max_k = max(k_values)

    for query_index in range(len(dataframe)):
        start_time = time.perf_counter()

        similarities = embeddings @ embeddings[query_index]
        similarities[query_index] = -np.inf
        ranked_indices = np.argsort(similarities)[::-1][:max_k]

        retrieval_times.append(time.perf_counter() - start_time)

        query_label = labels[query_index]
        total_relevant = int(np.sum(labels == query_label) - 1)
        row = {"query_index": query_index, "query_label": query_label}

        for k in k_values:
            top_k_indices = ranked_indices[:k]
            relevant_retrieved = int(
                np.sum(labels[top_k_indices] == query_label)
            )

            precision = relevant_retrieved / k if k else 0.0
            recall = (
                relevant_retrieved / total_relevant
                if total_relevant > 0
                else 0.0
            )
            f1 = (
                2 * precision * recall / (precision + recall)
                if precision + recall > 0
                else 0.0
            )

            row[f"relevant_at_{k}"] = relevant_retrieved
            row[f"precision_at_{k}"] = precision
            row[f"recall_at_{k}"] = recall
            row[f"f1_at_{k}"] = f1

        rows.append(row)

    query_metrics_df = pd.DataFrame(rows)
    summary = {"number_of_queries": len(dataframe)}

    for k in k_values:
        summary[f"precision_at_{k}"] = query_metrics_df[
            f"precision_at_{k}"
        ].mean()
        summary[f"recall_at_{k}"] = query_metrics_df[
            f"recall_at_{k}"
        ].mean()
        summary[f"f1_at_{k}"] = query_metrics_df[
            f"f1_at_{k}"
        ].mean()

    summary["average_retrieval_time_ms"] = (
        np.mean(retrieval_times) * 1000
    )

    return query_metrics_df, pd.DataFrame([summary])


In [ ]:
query_metrics_df, retrieval_summary_df = evaluate_image_retrieval(
    dataframe=test_df,
    embeddings=gallery_embeddings,
    k_values=(5, 10),
    label_column="label",
)

display(retrieval_summary_df)


In [ ]:
query_index = np.random.randint(len(retrieval_df))
query_row = retrieval_df.iloc[query_index]

query_path = query_row["EuroSAT_RGB_path"]
query_label = query_row["label"]

print("Query index:", query_index)
print("Query path:", query_path)
print("Query label:", query_label)

results = retrieve_similar_images(
    query_path=query_path,
    gallery_df=retrieval_df,
    gallery_embeddings=gallery_embeddings,
    model=embedding_model,
    top_k=10,
)

display(results)

display_retrieval_grid(
    query_path=query_path,
    query_label=query_label,
    retrieval_results=results,
    columns=4,
)


In [ ]:
top5_results = retrieve_same_modal_query(
    query_index=query_index,
    dataframe=test_df,
    embeddings=gallery_embeddings,
    top_k=5,
)

display(top5_results)

metrics_at_5 = calculate_query_metrics(
    query_index=query_index,
    retrieval_results=top5_results,
    dataframe=test_df,
)

print("Same-modal RGB → RGB")
print("Query label:", metrics_at_5["query_label"])
print("Precision@5:", round(metrics_at_5["precision_at_k"], 4))
print("Recall@5:", round(metrics_at_5["recall_at_k"], 4))
print("F1-score@5:", round(metrics_at_5["f1_at_k"], 4))


In [ ]:
top10_results = retrieve_same_modal_query(
    query_index=query_index,
    dataframe=test_df,
    embeddings=gallery_embeddings,
    top_k=10,
)

display(top10_results)

metrics_at_10 = calculate_query_metrics(
    query_index=query_index,
    retrieval_results=top10_results,
    dataframe=test_df,
)

print("Same-modal RGB → RGB")
print("Query label:", metrics_at_10["query_label"])
print("Precision@10:", round(metrics_at_10["precision_at_k"], 4))
print("Recall@10:", round(metrics_at_10["recall_at_k"], 4))
print("F1-score@10:", round(metrics_at_10["f1_at_k"], 4))


In [ ]:
embedding_path = Path("/kaggle/working/optical_gallery_embeddings.npy")
metadata_path = Path("/kaggle/working/optical_gallery_metadata.csv")

np.save(embedding_path, gallery_embeddings)
retrieval_df.to_csv(metadata_path, index=False)

print(f"Embeddings saved to: {embedding_path}")
print(f"Metadata saved to: {metadata_path}")


In [ ]:
# End of notebook
